# STIV -- interactive survey builder

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/simreaney/STIV/blob/main/notebooks/STIV_Interactive.ipynb)

A guided, step-by-step way to measure river surface velocity from a drone video, with no config file or command line needed. Each step asks for one thing at a time -- the video, the scale, the cross-section, a few settings -- then runs the analysis and shows the results, including a map of flow arrows whose length is proportional to the velocity.

**First time here?** Choose "Try the demo river" in step 1 -- it renders a short synthetic clip with a *known* velocity profile, so you can see the whole workflow (and check STIV is installed correctly) before pointing it at real footage.

This wizard covers the common case: a drone hovering roughly overhead. For an oblique view (needs ground control points), or for scripting a batch of surveys, see the command-line tool described in the [README](https://github.com/simreaney/STIV#readme) instead -- it uses exactly the same analysis underneath.

## 1. Setup

Run the cell below first. It installs STIV from [GitHub](https://github.com/simreaney/STIV) if it is not installed already (a minute or so the first time). In Google Colab it also switches on the widget support the wizard needs and connects your Google Drive -- the easiest place to keep large drone videos, and where the results are saved, since Colab wipes its own disk when the session ends.

- **Running on your own computer for the first time?** After the install, reload the browser page once so the interactive picker can load, then carry on.
- **Working in a clone of the repository?** Install it with `pip install -e '.[notebook]'` from the repository root instead; the cell then leaves that install alone.

In [ ]:
# Setup -- run this cell first.
UPDATE = False       # True: replace the installed STIV with the latest from GitHub (not in a clone)
MOUNT_DRIVE = True   # Colab only: connect Google Drive, for videos and results

import importlib.util
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules
PACKAGE = "stiv[widgets] @ git+https://github.com/simreaney/STIV.git"


def pip(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *args])


if UPDATE or importlib.util.find_spec("stiv") is None:
    print("Installing STIV from GitHub...")
    if UPDATE:
        # STIV itself afresh, without reinstalling numpy & co. underneath a
        # running kernel; the second call then adds anything newly required.
        pip("--force-reinstall", "--no-deps", PACKAGE.replace("[widgets]", ""))
    pip(PACKAGE)
    importlib.invalidate_caches()
    if not IN_COLAB:
        print("Installed. Reload this browser page once so the interactive picker can load.")

if IN_COLAB:
    from google.colab import output

    output.enable_custom_widget_manager()   # ipympl and the file browser are third-party widgets
    if MOUNT_DRIVE:
        from google.colab import drive

        drive.mount("/content/drive")

import stiv

print(f"STIV {stiv.__version__} ready.")

## 2. The wizard

Work through the six steps in the panel. The **Your survey** box on the right keeps track of every choice, and the step names along the top let you jump back to anything already done.

- **Scale:** pick your drone from the list and type the flight altitude above the *water*. DJI spec sheets quote the *diagonal* field of view; the list converts it to the horizontal one the scale needs. If the drone saved a flight log (`.SRT`, same name as the video) it is found automatically and can fill in the altitude.
- **Cross-section:** click the two banks and a point downstream. The water is then outlined automatically -- check it and press Confirm.
- **Coming back later:** every run saves `config_used.yaml` in its results folder. Choose "Reopen a saved survey" in step 1 to change a setting and run again without re-clicking anything.

In [ ]:
from stiv.notebook_ui import launch

wizard = launch()

## After a run

Once step 6 shows results, everything is also on disk in the folder named at the bottom of that step (`velocity.csv`, the plots -- including `velocity_map.png`, the flow arrows -- `report.txt` and `config_used.yaml`). In Colab, use **Download results (.zip)**, or keep the folder on Google Drive.

The full result is also available in this notebook, if you want to explore it further -- for example the cell below. It is cleared when you start a new survey or load another video; after that, read the saved `velocity.csv` instead.

In [ ]:
# df = wizard.state.result.to_frame()
# df.plot(x="station_m", y="velocity_ms", yerr="velocity_std_ms")